# cuecard Pipeline Debugger

Practical debugging and iteration tool for the multi-stage retrieval pipeline.

**Pipeline stages:**
1. Dense retrieval (fastembed cosine similarity)
2. Sparse retrieval (BM25)
3. RRF fusion (reciprocal rank fusion)
4. LLM reranker (Gemma 4 E4B via llama-server)
5. Event affinity mask (tool_use vs workflow filtering)

**Sections:**
- [Setup](#Setup) -- Load config, index, model
- [Per-Stage Visualization](#Per-Stage-Visualization) -- See what each stage contributes
- [Quality Metrics Per Stage](#Quality-Metrics-Per-Stage) -- Where do correct rules get dropped?
- [Loss Analysis](#Loss-Analysis) -- Debug failing fixtures
- [Prompts](#Prompts) -- Inspect LLM prompts (reranker, expansion, affinity)
- [Config Experimentation](#Config-Experimentation) -- Tweak parameters, compare results
- [Batch Evaluation](#Batch-Evaluation) -- Run eval across fixture files

## Setup

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

# Project root
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src" / "cuecard").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Load embedding model and build index from the eval corpus
from fastembed import TextEmbedding
from cuecard.parser import parse_rules
from cuecard.indexer import build_index, load_rules_json
from cuecard.models import (
    ResolvedConfig, PipelineConfig, Index, AffinityIndex,
    KNOWN_HOOK_EVENTS, RankedResult,
)
from cuecard.affinity import load_affinity, build_event_mask
from cuecard.eval import load_fixtures, Fixture

# --- Configurable paths ---
CORPUS_PATH = str(PROJECT_ROOT / "eval" / "corpora" / "rules_global.txt")
FIXTURE_DIR = str(PROJECT_ROOT / "eval" / "fixtures")
CACHE_DIR = str(Path.home() / ".cuecard" / "index")

# --- Embedding model ---
MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = TextEmbedding(MODEL_NAME)
print(f"Loaded embedding model: {MODEL_NAME}")

# --- Parse rules and build index ---
rules = parse_rules((CORPUS_PATH,))
print(f"Parsed {len(rules)} rules from {CORPUS_PATH}")

# Try to load enriched index (with expansions) from cache
cached = load_rules_json(CACHE_DIR)
if cached is not None:
    cached_rules, cached_affinity = cached
    # Merge expansions from cache into parsed rules
    from cuecard.indexer import merge_rules_json
    rules = merge_rules_json(rules, cached_rules)
    print(f"Merged expansions from cache ({sum(len(r.expansions) for r in rules)} total)")

index = build_index(tuple(rules), {}, MODEL_NAME, model=model)
print(f"Index: {index.size} rules, {index.embeddings.shape[0]} embeddings, dim={index.dim}")

# --- Load affinity ---
affinity = load_affinity(CACHE_DIR)
if affinity is not None:
    print(f"Affinity loaded: {affinity}")
else:
    print("No affinity index found (event mask will be disabled)")

In [ ]:
# --- Minimal config for pipeline calls ---
# Change these to experiment with different parameters
TOP_K = 7
THRESHOLD = 0.30
FUSION_K = 10
LLM_CANDIDATES = 12
DEDUP_THRESHOLD = 0.95
QUERY_MAX_LENGTH = 500
LLM_ENDPOINT = "http://localhost:8081/v1"

class NotebookConfig:
    """Minimal config compatible with pipeline.run_pipeline()."""
    def __init__(self, **kwargs):
        defaults = dict(
            top_k=TOP_K, threshold=THRESHOLD, fusion_k=FUSION_K,
            llm_candidates=LLM_CANDIDATES, dedup_threshold=DEDUP_THRESHOLD,
            query_max_length=QUERY_MAX_LENGTH, sparse_enabled=True,
            pipeline=PipelineConfig(
                mode="llm-local",
                local_endpoint=LLM_ENDPOINT,
            ),
        )
        defaults.update(kwargs)
        for k, v in defaults.items():
            setattr(self, k, v)

config = NotebookConfig()
print("Config ready. LLM endpoint:", LLM_ENDPOINT)

In [ ]:
# --- Helper functions used throughout the notebook ---

def snippet(text, max_len=80):
    """Truncate text for table display."""
    return text[:max_len] + "..." if len(text) > max_len else text


def candidates_df(candidates, label=""):
    """Convert a list of ScoredCandidate or RankedResult to a DataFrame."""
    rows = []
    for i, c in enumerate(candidates):
        retriever = getattr(c, 'retriever', 'pipeline')
        rows.append({
            "rank": i + 1,
            "score": round(c.score, 4),
            "retriever": retriever,
            "rule": snippet(c.rule.text),
        })
    df = pd.DataFrame(rows)
    if label:
        df.attrs["name"] = label
    return df


def highlight_matches(df, expected_rules):
    """Style a DataFrame: green for expected rules, red for noise."""
    expected_snippets = {snippet(r) for r in expected_rules}
    def row_style(row):
        if row["rule"] in expected_snippets:
            return ["background-color: #d4edda"] * len(row)
        return ["background-color: #f8d7da"] * len(row)
    return df.style.apply(row_style, axis=1)


def print_stage_header(name, count_in, count_out, latency_ms=None):
    """Print a formatted stage header."""
    lat = f" ({latency_ms:.0f}ms)" if latency_ms else ""
    print(f"\n{'='*60}")
    print(f"  {name}: {count_in} in -> {count_out} out{lat}")
    print(f"{'='*60}")

## Per-Stage Visualization

Run a single query through each pipeline stage independently and see what each contributes.

In [ ]:
# --- Choose a query to debug ---
QUERY = "Bash: git commit -m 'fix auth bug'"
EVENT = "PreToolUse"
TOOL_NAME = "Bash"

# Expected rules (from a fixture, or manually specified)
EXPECTED_RULES = [
    "When running git add or git commit, or writing config/.env files: never include secrets \u2014 once pushed, secrets are in the history forever and require rotation across all environments",
    "When running git commit or git add on Python files: run ruff check and mypy first \u2014 type errors and lint violations caught at commit time cost 10 minutes to fix, but in production they cost hours to debug",
]

print(f"Query: {QUERY}")
print(f"Event: {EVENT}")
print(f"Expected rules: {len(EXPECTED_RULES)}")
for r in EXPECTED_RULES:
    print(f"  - {snippet(r, 100)}")

In [ ]:
# --- Stage 1a: Dense retrieval ---
from cuecard.retrievers.dense import DenseRetriever

dense = DenseRetriever(
    model=model,
    dedup_threshold=config.dedup_threshold,
    max_query_length=config.query_max_length,
)

# Build event mask if affinity available
event_mask = None
if affinity is not None and EVENT:
    event_mask = build_event_mask(index, affinity, EVENT, tool_name=TOOL_NAME)
    masked_count = int((~event_mask).sum())
    print(f"Event mask: {masked_count} embeddings masked out of {len(event_mask)}")

t0 = time.monotonic()
dense_results = dense.retrieve(
    QUERY, index,
    top_k=config.llm_candidates,
    threshold=0.25,
    mask=event_mask,
)
dense_ms = (time.monotonic() - t0) * 1000

print_stage_header("Dense Retrieval", index.size, len(dense_results), dense_ms)
df_dense = candidates_df(dense_results, "dense")
display(highlight_matches(df_dense, EXPECTED_RULES))

In [ ]:
# --- Stage 1b: Sparse retrieval (BM25) ---
from cuecard.retrievers.sparse import SparseRetriever

sparse = SparseRetriever()
t0 = time.monotonic()
sparse_results = sparse.retrieve(
    QUERY, index,
    top_k=config.llm_candidates,
    threshold=0.0,
    mask=event_mask,
)
sparse_ms = (time.monotonic() - t0) * 1000

print_stage_header("Sparse Retrieval (BM25)", index.size, len(sparse_results), sparse_ms)
df_sparse = candidates_df(sparse_results, "sparse")
display(highlight_matches(df_sparse, EXPECTED_RULES))

In [ ]:
# --- Stage 1c: RRF Fusion ---
from cuecard.retrievers import fuse, ScoredCandidate

t0 = time.monotonic()
fused_results = fuse(
    [dense_results, sparse_results],
    k=config.fusion_k,
    top_k=config.llm_candidates,
)
fusion_ms = (time.monotonic() - t0) * 1000

# Track which retriever contributed each candidate
dense_texts = {c.rule.text for c in dense_results}
sparse_texts = {c.rule.text for c in sparse_results}

fused_rows = []
for i, c in enumerate(fused_results):
    in_dense = c.rule.text in dense_texts
    in_sparse = c.rule.text in sparse_texts
    source = "both" if (in_dense and in_sparse) else ("dense" if in_dense else "sparse")
    fused_rows.append({
        "rank": i + 1,
        "rrf_score": round(c.score, 4),
        "source": source,
        "rule": snippet(c.rule.text),
    })

print_stage_header(
    f"RRF Fusion (k={config.fusion_k})",
    len(dense_results) + len(sparse_results),
    len(fused_results),
    fusion_ms,
)
df_fused = pd.DataFrame(fused_rows)
display(highlight_matches(df_fused, EXPECTED_RULES))

# Summary: unique contributions
dense_only = dense_texts - sparse_texts
sparse_only = sparse_texts - dense_texts
print(f"\nDense-only candidates: {len(dense_only)}")
print(f"Sparse-only candidates: {len(sparse_only)}")
print(f"In both: {len(dense_texts & sparse_texts)}")

In [ ]:
# --- Stage 2: LLM Reranker ---
# Requires llama-server running on LLM_ENDPOINT
from cuecard.llm_reranker import rerank_llm

# Convert fused ScoredCandidates to RankedResults for the LLM reranker
llm_input = [
    RankedResult(rule=c.rule, score=c.score)
    for c in fused_results
]

try:
    t0 = time.monotonic()
    llm_results = rerank_llm(
        llm_input, QUERY,
        backend="local",
        endpoint=LLM_ENDPOINT,
        top_k=config.top_k,
    )
    llm_ms = (time.monotonic() - t0) * 1000

    print_stage_header("LLM Reranker", len(llm_input), len(llm_results), llm_ms)

    # Show included rules
    included_texts = {r.rule.text for r in llm_results}
    excluded = [r for r in llm_input if r.rule.text not in included_texts]

    print("\nINCLUDED:")
    df_llm = candidates_df(llm_results, "llm")
    display(highlight_matches(df_llm, EXPECTED_RULES))

    print(f"\nEXCLUDED ({len(excluded)} rules):")
    for r in excluded:
        marker = "[MISS]" if r.rule.text in EXPECTED_RULES else "      "
        print(f"  {marker} {snippet(r.rule.text, 100)}")

except Exception as e:
    print(f"LLM reranker unavailable: {e}")
    print("Start llama-server to enable this stage.")
    llm_results = llm_input[:config.top_k]
    llm_ms = 0

## Quality Metrics Per Stage

For a given fixture, track where correct rules appear and disappear through the pipeline.

In [ ]:
# --- Ceiling analysis: which correct rules are present at each stage? ---
from cuecard.eval import precision_at_k, recall_at_k, noise_ratio, quality_score

relevant = set(EXPECTED_RULES)
is_negative = len(relevant) == 0

stages_data = [
    ("Dense (top-k)", [c.rule.text for c in dense_results]),
    ("Sparse (top-k)", [c.rule.text for c in sparse_results]),
    ("RRF Fused", [c.rule.text for c in fused_results]),
    ("LLM Reranker", [r.rule.text for r in llm_results]),
]

ceiling_rows = []
for stage_name, retrieved in stages_data:
    hits = [r for r in relevant if r in retrieved]
    missed = [r for r in relevant if r not in retrieved]
    ceiling_rows.append({
        "stage": stage_name,
        "candidates": len(retrieved),
        "correct_found": len(hits),
        "correct_missed": len(missed),
        "precision": round(precision_at_k(retrieved, relevant), 3),
        "recall": round(recall_at_k(retrieved, relevant), 3),
        "noise": round(noise_ratio(retrieved, relevant), 3),
        "F2": round(quality_score(retrieved, relevant, is_negative), 3),
    })

df_ceiling = pd.DataFrame(ceiling_rows)
display(Markdown("### Pipeline Ceiling Analysis"))
display(df_ceiling)

# Show where each correct rule first appears and where it gets dropped
display(Markdown("### Per-Rule Tracking"))
for rule_text in relevant:
    print(f"\nRule: {snippet(rule_text, 100)}")
    for stage_name, retrieved in stages_data:
        present = rule_text in retrieved
        rank = retrieved.index(rule_text) + 1 if present else None
        status = f"rank {rank}" if present else "MISSING"
        marker = "  +" if present else "  X"
        print(f"{marker} {stage_name}: {status}")

## Loss Analysis

Debug failing fixtures: why did the pipeline miss expected rules or include false positives?

In [ ]:
# --- Load a fixture file and find failures ---
FIXTURE_FILE = "basic.json"  # Change to: workflow.json, post_tool_use.json, stop.json, etc.

fixtures = load_fixtures(str(PROJECT_ROOT / "eval" / "fixtures" / FIXTURE_FILE))
print(f"Loaded {len(fixtures)} fixtures from {FIXTURE_FILE}")
print(f"Events: {set(f.event for f in fixtures)}")
print(f"Tiers: {dict(sorted({t: sum(1 for f in fixtures if f.difficulty == t) for t in set(f.difficulty for f in fixtures)}.items()))}")

In [ ]:
# --- Run the full pipeline on one fixture and analyze ---
from cuecard.pipeline import run_pipeline

# Pick a fixture to debug (by index or by id)
FIXTURE_IDX = 0  # Change this, or use the search below
# FIXTURE_IDX = next(i for i, f in enumerate(fixtures) if f.id == "git-commit-secrets")

fx = fixtures[FIXTURE_IDX]
print(f"Fixture: {fx.id}")
print(f"Query: {fx.query}")
print(f"Event: {fx.event} | Difficulty: {fx.difficulty}")
print(f"Expected ({len(fx.should_match)}):")
for r in fx.should_match:
    print(f"  + {snippet(r, 100)}")
if fx.should_not_match:
    print(f"Anti-relevant ({len(fx.should_not_match)}):")
    for r in fx.should_not_match:
        print(f"  - {snippet(r, 100)}")

In [ ]:
# --- Run full pipeline on the fixture ---
try:
    pipeline_result = run_pipeline(
        fx.query, index, config,
        embedding_model=model,
        mode="llm-local",
        event=fx.event,
        affinity=affinity,
    )
    retrieved_texts = [r.rule.text for r in pipeline_result.results]
    mode_label = "llm-local"
except Exception as e:
    print(f"LLM unavailable ({e}), falling back to embedding mode")
    pipeline_result = run_pipeline(
        fx.query, index, config,
        embedding_model=model,
        mode="embedding",
        event=fx.event,
        affinity=affinity,
    )
    retrieved_texts = [r.rule.text for r in pipeline_result.results]
    mode_label = "embedding"

# Compute metrics
fx_relevant = set(fx.should_match)
fx_anti = set(fx.should_not_match)
fx_negative = fx.difficulty == "negative"

p = precision_at_k(retrieved_texts, fx_relevant)
r = recall_at_k(retrieved_texts, fx_relevant)
n = noise_ratio(retrieved_texts, fx_relevant)
f2 = quality_score(retrieved_texts, fx_relevant, fx_negative)

print(f"\nMode: {mode_label}")
print(f"Retrieved: {len(retrieved_texts)} rules")
print(f"Precision: {p:.3f} | Recall: {r:.3f} | Noise: {n:.3f} | F2: {f2:.3f}")

# Stage traces
for stage in pipeline_result.stages:
    err = f" [ERROR: {stage.error}]" if stage.error else ""
    print(f"  {stage.stage}: {stage.input_count}->{stage.output_count} ({stage.latency_ms:.0f}ms){err}")
if pipeline_result.event_mask_applied:
    print(f"  Event mask: {pipeline_result.rules_masked} rules masked")

In [ ]:
# --- Loss breakdown ---
display(Markdown("### Retrieved Rules"))
rows = []
for i, r in enumerate(pipeline_result.results):
    is_correct = r.rule.text in fx_relevant
    is_anti = r.rule.text in fx_anti
    label = "CORRECT" if is_correct else ("ANTI" if is_anti else "noise")
    rows.append({
        "rank": i + 1,
        "score": round(r.score, 4),
        "label": label,
        "rule": snippet(r.rule.text, 100),
    })
df_retrieved = pd.DataFrame(rows)

def label_color(row):
    if row["label"] == "CORRECT":
        return ["background-color: #d4edda"] * len(row)
    if row["label"] == "ANTI":
        return ["background-color: #f5c6cb"] * len(row)
    return ["background-color: #fff3cd"] * len(row)

display(df_retrieved.style.apply(label_color, axis=1))

# Show missed rules
missed = [r for r in fx.should_match if r not in retrieved_texts]
if missed:
    display(Markdown(f"### Missed Rules ({len(missed)})"))
    for r in missed:
        print(f"  MISSED: {snippet(r, 120)}")
else:
    display(Markdown("### All expected rules retrieved"))

In [ ]:
# --- Embedding similarity analysis for missed rules ---
from cuecard.retriever import normalize_query
from cuecard._math import l2_normalize

query_normalized = normalize_query(fx.query)
query_vec = np.array(list(model.query_embed([query_normalized])), dtype=np.float32)
query_vec = l2_normalize(query_vec)

# Score every rule against the query (raw cosine, no threshold)
all_scores = (index.embeddings @ query_vec.T).flatten()

# Parent collapse: max score per parent
parent_scores = np.full(len(index.rules), -np.inf, dtype=np.float64)
np.maximum.at(parent_scores, list(index.rule_map), all_scores)

display(Markdown("### Embedding Similarity: Query vs All Rules"))

sim_rows = []
for i, rule in enumerate(index.rules):
    is_expected = rule.text in fx_relevant
    is_retrieved = rule.text in retrieved_texts
    label = ""
    if is_expected and is_retrieved:
        label = "CORRECT+RETRIEVED"
    elif is_expected:
        label = "CORRECT+MISSED"
    elif is_retrieved:
        label = "NOISE"
    sim_rows.append({
        "rule_idx": i,
        "cosine": round(float(parent_scores[i]), 4),
        "label": label,
        "rule": snippet(rule.text, 80),
    })

df_sim = pd.DataFrame(sim_rows)
# Show only labeled rows, sorted by score
df_labeled = df_sim[df_sim["label"] != ""].sort_values("cosine", ascending=False)
display(df_labeled.reset_index(drop=True))

# Compare: best false positive vs worst correct rule
correct_scores = [float(parent_scores[i]) for i, r in enumerate(index.rules) if r.text in fx_relevant]
noise_scores = [float(parent_scores[i]) for i, r in enumerate(index.rules) if r.text in retrieved_texts and r.text not in fx_relevant]

if correct_scores:
    print(f"\nCorrect rule scores: min={min(correct_scores):.4f}, max={max(correct_scores):.4f}")
if noise_scores:
    print(f"Top false positive score: {max(noise_scores):.4f}")
    if correct_scores:
        gap = min(correct_scores) - max(noise_scores)
        print(f"Gap (worst correct - best noise): {gap:.4f}")

## Prompts

Inspect the exact prompts sent to the LLM at each stage.

In [ ]:
# --- LLM Reranker Prompt ---
# Shows exactly what the reranker LLM receives for the current query

import secrets as _secrets
from cuecard.llm_reranker import _build_prompt, _SYSTEM_PROMPT_TEMPLATE

# Build prompt from the fused candidates (same input as rerank_llm)
reranker_candidates = [
    RankedResult(rule=c.rule, score=c.score)
    for c in fused_results
]

nonce = _secrets.token_hex(6)
reranker_system, reranker_user = _build_prompt(reranker_candidates, QUERY, nonce)

display(Markdown("### LLM Reranker: System Prompt"))
print(reranker_system)

display(Markdown("### LLM Reranker: User Prompt"))
print(reranker_user)

print(f"\n--- Prompt stats ---")
print(f"System prompt: {len(reranker_system)} chars")
print(f"User prompt: {len(reranker_user)} chars")
print(f"Candidate rules: {len(reranker_candidates)}")
print(f"Nonce: {nonce}")

In [ ]:
# --- Expansion Prompt ---
# Shows the prompt used to generate expansions for a specific rule

from cuecard.expander import _build_expansion_prompt

# Pick a rule to see its expansion prompt
EXPANSION_RULE_IDX = 0  # Index into index.rules
EXPANSION_EVENT_TYPE = "PreToolUse"  # or "UserPromptSubmit" for workflow rules

expansion_rule = index.rules[EXPANSION_RULE_IDX]
exp_nonce = _secrets.token_hex(6)
exp_system, exp_user = _build_expansion_prompt(
    expansion_rule.text, exp_nonce, event_type=EXPANSION_EVENT_TYPE,
)

display(Markdown(f"### Expansion Prompt for Rule {EXPANSION_RULE_IDX}"))
print(f"Rule text: {snippet(expansion_rule.text, 120)}")
print(f"Event type: {EXPANSION_EVENT_TYPE}")

if expansion_rule.expansions:
    print(f"\nCurrent expansions ({len(expansion_rule.expansions)}):")
    for exp in expansion_rule.expansions:
        print(f"  - {exp}")

display(Markdown("#### System Prompt"))
print(exp_system)

display(Markdown("#### User Prompt"))
print(exp_user)

print(f"\n--- Prompt stats ---")
print(f"System prompt: {len(exp_system)} chars")
print(f"User prompt: {len(exp_user)} chars")

In [ ]:
# --- Affinity Classification Prompt ---
# Shows the prompt used to classify a rule as tool_use/workflow/both

from cuecard.affinity import _build_affinity_prompt

# Pick a rule to see its affinity prompt
AFFINITY_RULE_IDX = 0  # Index into index.rules

aff_rule = index.rules[AFFINITY_RULE_IDX]
aff_nonce = _secrets.token_hex(6)
aff_system, aff_user = _build_affinity_prompt(
    aff_rule.text, aff_nonce,
    explicit_events=aff_rule.events,
    explicit_tools=aff_rule.tools,
)

display(Markdown(f"### Affinity Prompt for Rule {AFFINITY_RULE_IDX}"))
print(f"Rule text: {snippet(aff_rule.text, 120)}")

# Show current affinity if available
if affinity is not None:
    ra = affinity.get(aff_rule)
    if ra is not None:
        print(f"Current affinity: events={sorted(ra.events)}, source={ra.source}")
        if ra.reasoning:
            print(f"Reasoning: {ra.reasoning}")

display(Markdown("#### System Prompt"))
print(aff_system)

display(Markdown("#### User Prompt"))
print(aff_user)

print(f"\n--- Prompt stats ---")
print(f"System prompt: {len(aff_system)} chars")
print(f"User prompt: {len(aff_user)} chars")

## Config Experimentation

Run the same fixture through the pipeline with different configs and compare side by side.

In [ ]:
# --- Compare different pipeline configs on a single fixture ---

EXPERIMENT_QUERY = fx.query  # Uses the fixture from Loss Analysis section
EXPERIMENT_EVENT = fx.event
EXPERIMENT_EXPECTED = set(fx.should_match)
EXPERIMENT_NEGATIVE = fx.difficulty == "negative"

# Define configs to compare
experiments = {
    "default (top_k=7, fusion_k=10)": dict(
        top_k=7, threshold=0.30, fusion_k=10, llm_candidates=12,
    ),
    "wider recall (top_k=12, fusion_k=10)": dict(
        top_k=12, threshold=0.25, fusion_k=10, llm_candidates=20,
    ),
    "narrow (top_k=5, fusion_k=60)": dict(
        top_k=5, threshold=0.35, fusion_k=60, llm_candidates=12,
    ),
    "aggressive (top_k=15, threshold=0.20)": dict(
        top_k=15, threshold=0.20, fusion_k=10, llm_candidates=25,
    ),
}

comparison_rows = []
for label, params in experiments.items():
    exp_config = NotebookConfig(**params)
    try:
        result = run_pipeline(
            EXPERIMENT_QUERY, index, exp_config,
            embedding_model=model,
            mode="embedding",
            event=EXPERIMENT_EVENT,
            affinity=affinity,
        )
        texts = [r.rule.text for r in result.results]
        comparison_rows.append({
            "config": label,
            "retrieved": len(texts),
            "precision": round(precision_at_k(texts, EXPERIMENT_EXPECTED), 3),
            "recall": round(recall_at_k(texts, EXPERIMENT_EXPECTED), 3),
            "noise": round(noise_ratio(texts, EXPERIMENT_EXPECTED), 3),
            "F2": round(quality_score(texts, EXPERIMENT_EXPECTED, EXPERIMENT_NEGATIVE), 3),
        })
    except Exception as e:
        comparison_rows.append({
            "config": label, "retrieved": 0,
            "precision": 0, "recall": 0, "noise": 0, "F2": 0,
        })
        print(f"  {label}: FAILED ({e})")

display(Markdown(f"### Config Comparison: `{fx.id}`"))
display(pd.DataFrame(comparison_rows))

In [ ]:
# --- Compare embedding-only vs LLM mode on the same fixture ---

mode_rows = []
for mode_name in ["embedding", "llm-local"]:
    try:
        result = run_pipeline(
            EXPERIMENT_QUERY, index, config,
            embedding_model=model,
            mode=mode_name,
            event=EXPERIMENT_EVENT,
            affinity=affinity,
        )
        texts = [r.rule.text for r in result.results]
        total_ms = sum(s.latency_ms for s in result.stages)
        mode_rows.append({
            "mode": mode_name,
            "retrieved": len(texts),
            "precision": round(precision_at_k(texts, EXPERIMENT_EXPECTED), 3),
            "recall": round(recall_at_k(texts, EXPERIMENT_EXPECTED), 3),
            "noise": round(noise_ratio(texts, EXPERIMENT_EXPECTED), 3),
            "F2": round(quality_score(texts, EXPERIMENT_EXPECTED, EXPERIMENT_NEGATIVE), 3),
            "latency_ms": round(total_ms),
        })
    except Exception as e:
        mode_rows.append({
            "mode": mode_name, "retrieved": 0,
            "precision": 0, "recall": 0, "noise": 0, "F2": 0,
            "latency_ms": 0,
        })
        print(f"  {mode_name}: {e}")

display(Markdown("### Mode Comparison: embedding vs llm-local"))
display(pd.DataFrame(mode_rows))

## Batch Evaluation

Run the eval framework across fixture files with tqdm progress.

In [ ]:
# --- Batch eval across all fixture files ---
from cuecard.eval import run_eval, evaluate_per_event, EvalSummary

EVAL_MODE = "embedding"  # Change to "llm-local" if llama-server is running
SAMPLE_RATIO = 0.2  # Use 1.0 for full eval, 0.2 for quick iteration

FIXTURE_FILES = [
    "basic.json",
    "workflow.json",
    "post_tool_use.json",
    "stop.json",
    "subagent_start.json",
]

all_fixtures = []
for fname in FIXTURE_FILES:
    fpath = str(PROJECT_ROOT / "eval" / "fixtures" / fname)
    try:
        fxs = load_fixtures(fpath)
        all_fixtures.extend(fxs)
        print(f"  {fname}: {len(fxs)} fixtures")
    except Exception as e:
        print(f"  {fname}: FAILED ({e})")

print(f"\nTotal: {len(all_fixtures)} fixtures")
print(f"Mode: {EVAL_MODE} | Sample ratio: {SAMPLE_RATIO}")

In [ ]:
# --- Run eval ---
corpus_override = (CORPUS_PATH,)

summary = run_eval(
    all_fixtures,
    corpus_dir=str(PROJECT_ROOT / "eval" / "corpora"),
    model_name=MODEL_NAME,
    model=model,
    top_k=TOP_K,
    threshold=THRESHOLD,
    mode=EVAL_MODE if EVAL_MODE != "embedding" else None,
    corpus_override=corpus_override,
    sample_ratio=SAMPLE_RATIO,
    affinity=affinity,
)

print(f"\nEval complete: {summary.fixture_count} fixtures evaluated")

In [ ]:
# --- Aggregate metrics ---
display(Markdown("### Aggregate Metrics"))

agg_rows = [
    {"metric": "F2 (mean quality)", "value": f"{summary.mean_quality:.3f}"},
    {"metric": "Positive Quality", "value": f"{summary.positive_quality:.3f}"},
    {"metric": "Positive Recall", "value": f"{summary.positive_recall:.3f}"},
    {"metric": "Mean Noise", "value": f"{summary.mean_noise_ratio:.3f}"},
    {"metric": "Negative Silence", "value": f"{summary.negative_silence_rate:.3f}"},
    {"metric": "Mean Retrieved", "value": f"{summary.mean_retrieved_count:.1f}"},
    {"metric": "Latency p50", "value": f"{summary.latency_p50_ms:.0f} ms"},
    {"metric": "Latency p95", "value": f"{summary.latency_p95_ms:.0f} ms"},
]
display(pd.DataFrame(agg_rows))

In [ ]:
# --- Per-tier breakdown ---
display(Markdown("### Per-Tier Breakdown"))

tier_rows = []
for ts in summary.per_tier:
    tier_rows.append({
        "tier": ts.tier,
        "count": ts.count,
        "F2": round(ts.mean_quality, 3),
        "precision": round(ts.mean_precision, 3),
        "recall": round(ts.mean_recall, 3),
        "noise": round(ts.mean_noise_ratio, 3),
        "silence": round(ts.silence_rate, 3),
    })
display(pd.DataFrame(tier_rows))

In [ ]:
# --- Per-event breakdown ---
display(Markdown("### Per-Event Breakdown"))

# Build fixture list matching the sampled eval results
eval_fixture_ids = {r.fixture_id for r in summary.per_fixture}
eval_fixtures = [f for f in all_fixtures if f.id in eval_fixture_ids]

per_event = evaluate_per_event(list(summary.per_fixture), eval_fixtures)
event_rows = []
for em in per_event:
    event_rows.append({
        "event": em.event,
        "count": em.fixture_count,
        "F2": round(em.quality, 3),
        "PosRecall": round(em.positive_recall, 3),
        "Noise": round(em.noise_ratio, 3),
        "NegSilence": round(em.negative_silence, 3),
    })
display(pd.DataFrame(event_rows))

In [ ]:
# --- Worst fixtures (lowest F2) ---
display(Markdown("### Worst Fixtures (lowest F2)"))

sorted_fixtures = sorted(summary.per_fixture, key=lambda r: r.quality_score)
worst_rows = []
for r in sorted_fixtures[:15]:
    worst_rows.append({
        "id": r.fixture_id,
        "difficulty": r.difficulty,
        "F2": round(r.quality_score, 3),
        "precision": round(r.precision_at_k, 3),
        "recall": round(r.recall_at_k, 3),
        "noise": round(r.noise_ratio, 3),
        "retrieved": r.retrieved_count,
        "query": snippet(r.query, 60),
    })
display(pd.DataFrame(worst_rows))

In [ ]:
# --- False positive analysis: most common noise rules ---
display(Markdown("### Most Common Noise Rules"))

noise_counter = {}
for r in summary.per_fixture:
    fx_match = next((f for f in all_fixtures if f.id == r.fixture_id), None)
    if fx_match is None:
        continue
    relevant_set = set(fx_match.should_match)
    for retrieved_text in r.retrieved:
        if retrieved_text not in relevant_set:
            key = snippet(retrieved_text, 80)
            noise_counter[key] = noise_counter.get(key, 0) + 1

noise_sorted = sorted(noise_counter.items(), key=lambda x: x[1], reverse=True)
noise_df_rows = [{"count": c, "rule": r} for r, c in noise_sorted[:15]]
display(pd.DataFrame(noise_df_rows))